# Modelo de Recomendación basado en TF-IDF y Similitud del Coseno

## Introducción
Este sistema de recomendación utiliza un enfoque basado en el contenido mediante **TF-IDF (Term Frequency - Inverse Document Frequency)** y **similitud del coseno** para encontrar películas con características textuales similares. Se procesan las reseñas de los usuarios, eliminando palabras irrelevantes (stopwords) y generando representaciones numéricas que permiten medir la similitud entre películas.

## Justificación del Modelo
TF-IDF es una técnica eficaz para extraer términos relevantes de textos, otorgando mayor peso a palabras distintivas y penalizando términos frecuentes. Esto permite capturar información clave de las reseñas, evitando sesgos por palabras comunes. La similitud del coseno, aplicada sobre los vectores TF-IDF, mide qué tan cercanas son las películas en términos de su contenido textual.

## Implementación
1. **Preprocesamiento de Datos:** Se combinan los géneros, reseñas y sinopsis en una sola columna.
2. **Vectorización con TF-IDF:** Se eliminan stopwords en español y se transforma el texto en una matriz numérica.
3. **Cálculo de Similitud:** Se usa la similitud del coseno para comparar los vectores de características entre películas.
4. **Generación de Recomendaciones:** Se ordenan las películas según su similitud con la película consultada y se devuelven las más relevantes.

## Ventajas del Enfoque
- No requiere información explícita del usuario.
- Identifica similitudes basadas en contenido textual sin necesidad de etiquetas manuales.
- Escalable a grandes volúmenes de datos utilizando técnicas de optimización.

Este modelo es adecuado para sistemas de recomendación de contenido, especialmente cuando se dispone de descripciones detalladas en texto y se busca encontrar títulos similares en función del lenguaje utilizado en las reseñas y sinopsis.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import nltk
from nltk.corpus import stopwords

# Descargar las stopwords en español
nltk.download('stopwords')
spanish_stopwords = stopwords.words('spanish')

# Ruta del archivo Parquet
file_path = 'C:/Users/jugas/OneDrive/Escritorio/Movie recommender/MVP_sistema_recomendacion/proyecto/data/movies_filtrado.parquet'

# Leer el archivo Parquet
df_filtrado = pd.read_parquet(file_path)

# Crear una columna 'caracteristicas' combinando géneros, reseñas y sinopsis
df_filtrado['caracteristicas'] = df_filtrado['generos'].astype(str) + ' ' + \
                                  df_filtrado['reseñas'].fillna('') + ' ' + \
                                  df_filtrado['sinopsis'].fillna('')

# Convertir la columna 'caracteristicas' a tipo texto
df_filtrado['caracteristicas'] = df_filtrado['caracteristicas'].astype(str)

# Eliminar filas con valores faltantes en 'caracteristicas'
df_filtrado = df_filtrado.dropna(subset=['caracteristicas'])

# Asegurarse de que todas las entradas en la columna 'reseñas' sean cadenas de texto
df_filtrado['reseñas'] = df_filtrado['reseñas'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

# Usamos TfidfVectorizer con las stopwords en español para procesar las reseñas
tfidf = TfidfVectorizer(stop_words=spanish_stopwords)
tfidf_matrix = tfidf.fit_transform(df_filtrado['reseñas'])

# Calcular la similitud del coseno entre las películas basándonos en las características
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Función para obtener las películas más similares
def obtener_recomendaciones(titulo_pelicula, cosine_sim=cosine_sim):
    # Verificar si el título de la película existe en el DataFrame
    if titulo_pelicula not in df_filtrado['titulo'].values:
        return f'La película "{titulo_pelicula}" no se encuentra en el DataFrame.'
    
    # Obtengo el índice de la película
    idx = df_filtrado.index[df_filtrado['titulo'] == titulo_pelicula].tolist()[0]
    
    # Obtengo las puntuaciones de similitud con todas las películas
    sim_scores = list(enumerate(cosine_sim[idx]))
    
    # Ordeno las películas basándome en las similitudes, y obtengo los 10 más similares
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    
    # Obtengo los índices de las películas más similares
    movie_indices = [i[0] for i in sim_scores]
    
    # Devuelvo las películas más similares
    return df_filtrado['titulo'].iloc[movie_indices]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jugas\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
# Función para obtener las películas más similares con título, sinopsis y puntuación
def obtener_recomendaciones(titulo_pelicula, df=df_filtrado, cosine_sim=cosine_sim, top_n=10):
    if titulo_pelicula not in df['titulo'].values:
        return f'La película "{titulo_pelicula}" no se encuentra en el DataFrame.'
    
    idx = df.index[df['titulo'] == titulo_pelicula].tolist()[0]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]
    
    # Devuelve un DataFrame con título, sinopsis y puntuación de las películas recomendadas
    return df.iloc[movie_indices][["titulo", "sinopsis", "puntuacion"]]

# Prueba con varias películas
movies_to_test = ["Matrix", "Origen", "Cars", "Harry Potter"]
for movie in movies_to_test:
    recommendations = obtener_recomendaciones(movie)
    
    if isinstance(recommendations, pd.DataFrame):  
        print(f"Película consultada: {movie}")
        print("Recomendaciones:")
        for _, row in recommendations.iterrows():
            print(f"- {row['titulo']}")
            print(f"  Sinopsis: {row['sinopsis']}")
            print(f"  Puntuación: {row['puntuacion']}")
            print()
    else:
        print(recommendations)

    print("\n")


Película consultada: Matrix
Recomendaciones:
- Origen
  Sinopsis: Dom Cobb es un ladrón hábil, el mejor de todos, especializado en el peligroso arte de extracción: el robo de secretos valiosos desde las profundidades del subconsciente durante el estado de sueño cuando la mente está más vulnerable. Esta habilidad excepcional de Cobb le ha hecho un jugador codiciado en el traicionero nuevo mundo de espionaje corporativo, pero al mismo tiempo, le ha convertido en un fugitivo internacional y ha tenido que sacrificar todo que le importaba. Ahora a Cobb se le ofrece una oportunidad para redimirse. Con un último trabajo podría recuperar su vida anterior, pero solamente si logra lo imposible.
  Puntuación: 8.369

- Transformers: El despertar de las bestias
  Sinopsis: Cuando surge una nueva amenaza capaz de destruir todo el planeta, Optimus Prime y los Autobots deben unirse a una poderosa facción conocida como los Maximals. Con el destino de la humanidad en juego, los humanos Noah y Elena hará